In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass
from typing import List, Callable, Any

@dataclass
class SensorProcesso:
    tag: str
    setor: str
    tipo: str
    valor: float
    unidade: str
    limite: float
    falha_comunicacao: bool = False

def FORALL(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return all(predicado(x) for x in dominio)

def EXISTS(dominio: List[Any], predicado: Callable[[Any], bool]) -> bool:
    return any(predicado(x) for x in dominio)

# Aplicação para a Planta de Biodiesel
rede_sensores = [
    SensorProcesso('AT-100A', 'Setor 100', 'GAS_METANOL', 2.1, '% LEL', 10.0),
    SensorProcesso('AT-100B', 'Setor 100', 'GAS_METANOL', 1.5, '% LEL', 10.0),
    # Falha de vazamento injetada propositalmente no AT-100C para o teste do EXISTS retornar True
    SensorProcesso('AT-100C', 'Setor 100', 'GAS_METANOL', 12.4, '% LEL', 10.0),
    SensorProcesso('TT-201A', 'Setor 200', 'TEMP_REATOR', 60.0, '°C', 65.0),
]

# Filtrando apenas os detectores de gás metanol para a varredura
detectores_gas = [s for s in rede_sensores if s.tipo == 'GAS_METANOL']

# Aplicando os quantificadores lógicos
existe_vazamento = EXISTS(detectores_gas, lambda s: s.valor >= s.limite)
todos_comunicando = FORALL(rede_sensores, lambda s: not s.falha_comunicacao)

print(f"1. Existe Vazamento de Gás Metanol (EXISTS): {existe_vazamento}")
print(f"2. Todos Sensores Comunicando (FORALL): {todos_comunicando}")

# Gerando a tabela visual
tabela = [{"Tag": s.tag, "Setor": s.setor, "Valor": f"{s.valor} {s.unidade}", "Limite": f"{s.limite} {s.unidade}", "Falha": s.valor >= s.limite} for s in rede_sensores]
print("\n" + formatar_tabela(tabela))

# Validação do teste
assert existe_vazamento is True

1. Existe Vazamento de Gás Metanol (EXISTS): True
2. Todos Sensores Comunicando (FORALL): True

Tag     | Setor     | Valor      | Limite     | Falha
--------+-----------+------------+------------+------
AT-100A | Setor 100 | 2.1 % LEL  | 10.0 % LEL | False
AT-100B | Setor 100 | 1.5 % LEL  | 10.0 % LEL | False
AT-100C | Setor 100 | 12.4 % LEL | 10.0 % LEL | True 
TT-201A | Setor 200 | 60.0 °C    | 65.0 °C    | False
